In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, 
    DoubleType, LongType, IntegerType, BooleanType, TimestampType
)

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SilverLayerDDL") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# ==========================================
# 1. Schema Definition: Table A (silver_price_features)
# ==========================================
schema_silver_price_features = StructType([
    StructField("symbol", StringType(), False),
    StructField("trade_date", DateType(), False),
    StructField("close", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("open", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("sma_200", DoubleType(), True),
    StructField("high_252", DoubleType(), True),
    StructField("mom_12_1", DoubleType(), True),
    StructField("return_12m", DoubleType(), True),
    StructField("avg_volume_50", DoubleType(), True),
    StructField("avg_turnover_20", DoubleType(), True),
    StructField("distance_from_52w_high", DoubleType(), True),
    StructField("above_sma_200", BooleanType(), True),
    StructField("price_days", IntegerType(), True),
    StructField("rs_rank", DoubleType(), True),
    StructField("sufficient_history_flag", BooleanType(), True),
    StructField("calculation_timestamp", TimestampType(), True),
    StructField("run_id", StringType(), True)
])

# Create Empty DataFrame with Schema
df_price_features = spark.createDataFrame([], schema_silver_price_features)

# Save to Delta Catalog / Data Lake
df_price_features.write \
    .format("delta") \
    .mode("ignore") \
    .partitionBy("trade_date") \
    .saveAsTable("silver_price_features")


# ==========================================
# 2. Schema Definition: Table B (silver_market_regime)
# ==========================================
schema_silver_market_regime = StructType([
    StructField("trade_date", DateType(), False),
    StructField("close", DoubleType(), True),
    StructField("sma_200", DoubleType(), True),
    StructField("above_sma_200", BooleanType(), True),
    StructField("market_regime", StringType(), True),
    StructField("calculation_timestamp", TimestampType(), True),
    StructField("run_id", StringType(), True)
])

# Create Empty DataFrame with Schema
df_market_regime = spark.createDataFrame([], schema_silver_market_regime)

# Save to Delta Catalog / Data Lake
df_market_regime.write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable("silver_market_regime")

StatementMeta(, 8eae7efc-2d36-482e-98b2-256a1f764c49, 3, Finished, Available, Finished, False)

In [1]:
# ============================================================
# NB04_Silver_Price_Features
# Purpose:
#   Convert bronze_price_ohlcv into momentum, trend,
#   liquidity and relative strength features.
#   Create silver_price_features + silver_market_regime
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F
from datetime import datetime
import uuid

RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

print(f"RUN_ID: {RUN_ID}")
print(f"RUN_TIMESTAMP: {RUN_TIMESTAMP}")


# ------------------------------------------------------------
# 1. Read config
# ------------------------------------------------------------

params_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load("Files/config/model_parameters.csv")
    .toPandas()
    .set_index("parameter")["value"]
)

MIN_PRICE_HISTORY_DAYS = int(params_df.get("min_price_history_days", 260))

print(f"MIN_PRICE_HISTORY_DAYS: {MIN_PRICE_HISTORY_DAYS}")




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 3, Finished, Available, Finished, False)

RUN_ID: ad31dbe3-d948-43df-9aed-3fb7c8d05f8d
RUN_TIMESTAMP: 2026-09-20 03:45:25
MIN_PRICE_HISTORY_DAYS: 260


In [2]:
# ------------------------------------------------------------
# 2. Load bronze price data
# ------------------------------------------------------------

price_raw = spark.read.table("bronze_price_ohlcv")

print("Bronze price rows:")
print(price_raw.count())
price_raw.printSchema()



StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 4, Finished, Available, Finished, False)

Bronze price rows:
655233
root
 |-- symbol: string (nullable = true)
 |-- yahoo_symbol: string (nullable = true)
 |-- trade_date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- source: string (nullable = true)
 |-- price_adjusted_flag: boolean (nullable = true)
 |-- retrieved_at: timestamp (nullable = true)
 |-- run_id: string (nullable = true)



In [3]:

# ------------------------------------------------------------
# 3. Clean and deduplicate price data
# ------------------------------------------------------------

price_clean = (
    price_raw
    .withColumn("trade_date", F.to_date("trade_date"))
    .withColumn("open", F.col("open").cast("double"))
    .withColumn("high", F.col("high").cast("double"))
    .withColumn("low", F.col("low").cast("double"))
    .withColumn("close", F.col("close").cast("double"))
    .withColumn("volume", F.col("volume").cast("long"))
    .filter(F.col("trade_date").isNotNull())
    .filter(F.col("symbol").isNotNull())
    .filter(F.col("close").isNotNull())
    .filter(F.col("close") > 0)
    .dropDuplicates(["symbol", "trade_date"])
)

# Load universe eligibility flag
universe = spark.sql("""
    SELECT
        symbol,
        instrument_type,
        eligible_for_stock_screen
    FROM bronze_universe_snapshot
""")

price_with_universe = (
    price_clean.join(
        universe,
        on="symbol",
        how="left"
    )
)

print("Clean price rows:")
print(price_with_universe.count())




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 5, Finished, Available, Finished, False)

Clean price rows:
655233


In [4]:
# ------------------------------------------------------------
# 4. Define rolling windows
# ------------------------------------------------------------

w_ordered = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
)

w_cumulative = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
    .rowsBetween(Window.unboundedPreceding, 0)
)

w_20 = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
    .rowsBetween(-19, 0)
)

w_50 = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
    .rowsBetween(-49, 0)
)

w_200 = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
    .rowsBetween(-199, 0)
)

w_252 = (
    Window
    .partitionBy("symbol")
    .orderBy("trade_date")
    .rowsBetween(-251, 0)
)




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 6, Finished, Available, Finished, False)

In [5]:
# ------------------------------------------------------------
# 5. Calculate rolling price features
# ------------------------------------------------------------

price_features = (
    price_with_universe
    .withColumn("price_days", F.count("close").over(w_cumulative))
    .withColumn("sma_200", F.avg("close").over(w_200))
    .withColumn("high_252", F.max("high").over(w_252))
    .withColumn("avg_volume_50", F.avg("volume").over(w_50))
    .withColumn(
        "avg_turnover_20",
        F.avg(F.col("close") * F.col("volume")).over(w_20)
    )
    .withColumn("close_lag_21", F.lag("close", 21).over(w_ordered))
    .withColumn("close_lag_252", F.lag("close", 252).over(w_ordered))
)

price_features = (
    price_features
    .withColumn(
        "mom_12_1",
        F.when(
            (F.col("close_lag_21").isNotNull()) &
            (F.col("close_lag_252").isNotNull()) &
            (F.col("close_lag_252") > 0),
            (F.col("close_lag_21") / F.col("close_lag_252") - 1)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "return_12m",
        F.when(
            (F.col("close_lag_252").isNotNull()) &
            (F.col("close_lag_252") > 0),
            (F.col("close") / F.col("close_lag_252") - 1)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "distance_from_52w_high",
        F.when(
            (F.col("high_252").isNotNull()) &
            (F.col("high_252") > 0),
            (F.col("close") / F.col("high_252") - 1)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "above_sma_200",
        F.when(
            F.col("sma_200").isNotNull(),
            F.col("close") > F.col("sma_200")
        ).otherwise(F.lit(None).cast("boolean"))
    )
    .withColumn(
        "sufficient_history_flag",
        (F.col("price_days") >= MIN_PRICE_HISTORY_DAYS) &
        (F.col("mom_12_1").isNotNull())
    )
)




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 7, Finished, Available, Finished, False)

In [6]:
# ------------------------------------------------------------
# 6. Calculate Relative Strength Rank
# ------------------------------------------------------------

# RS Rank is calculated only across eligible equities.
# Nifty 50 and ineligible stocks get NULL RS Rank.

w_rs = (
    Window
    .partitionBy("trade_date")
    .orderBy("mom_12_1")
)

price_features = (
    price_features
    .withColumn(
        "rs_rank_raw",
        F.when(
            (F.col("instrument_type") == "EQUITY") &
            (F.col("eligible_for_stock_screen") == True) &
            (F.col("mom_12_1").isNotNull()),
            F.percent_rank().over(w_rs) * 100
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("rs_rank", F.round("rs_rank_raw", 2))
    .drop("rs_rank_raw", "close_lag_21", "close_lag_252")
    .withColumn("calculation_timestamp", F.lit(RUN_TIMESTAMP))
    .withColumn("run_id", F.lit(RUN_ID))
)




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 8, Finished, Available, Finished, False)

In [7]:
# ------------------------------------------------------------
# 7. Create silver_price_features
# ------------------------------------------------------------

silver_price = price_features.filter(
    F.col("instrument_type") == "EQUITY"
)

silver_price_selected = silver_price.select(
    "symbol",
    "trade_date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "sma_200",
    "high_252",
    "mom_12_1",
    "return_12m",
    "avg_volume_50",
    "avg_turnover_20",
    "distance_from_52w_high",
    "above_sma_200",
    "price_days",
    "rs_rank",
    "sufficient_history_flag",
    "eligible_for_stock_screen",
    "calculation_timestamp",
    "run_id"
)

(
    silver_price_selected.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_price_features")
)

print("Saved: silver_price_features")
print(silver_price_selected.count())




StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 9, Finished, Available, Finished, False)

Saved: silver_price_features
653330


In [8]:
# ------------------------------------------------------------
# 8. Create silver_market_regime
# ------------------------------------------------------------

market_regime = (
    price_features
    .filter(F.col("symbol") == "NIFTY50")
    .select(
        "trade_date",
        "close",
        "sma_200",
        "above_sma_200",
        "calculation_timestamp",
        "run_id"
    )
    .withColumn(
        "market_regime",
        F.when(
            F.col("above_sma_200") == True,
            F.lit("RISK_ON")
        ).when(
            F.col("above_sma_200") == False,
            F.lit("RISK_OFF")
        ).otherwise(F.lit("UNKNOWN"))
    )
)

(
    market_regime.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_market_regime")
)

print("Saved: silver_market_regime")
print(market_regime.count())

StatementMeta(, 9acb7d29-06bf-4cc8-b9c2-98cdfdbe4211, 10, Finished, Available, Finished, False)

Saved: silver_market_regime
1903
